# Required ML Reporting Contract Gate

Fails closed before Reporting publication when a required ML Delta table is absent, empty, incompatible, temporally incomplete, or violates its grain and value constraints. It never creates or repairs output tables.

It also checks that:
- every table has one source as-of, the six agree (within `MAX_AS_OF_SKEW_HOURS`), and none is older than the current Silver as-of (the latest `fact_receipts.event_ts`) by more than `MAX_SOURCE_LAG_HOURS`, or newer than it;
- every store, product and customer key exists in its Silver dimension;
- category columns hold only the values the semantic model's measures filter on (`HIGH` stockout risk, `High`/`Very High` churn, ...);
- recorded quality metrics meet their floor: demand `mape` must beat the all-zero forecast (`MAX_FORECAST_MAPE`, default 1.0).


In [ ]:
from functools import reduce
import os

from pyspark.sql import functions as F
from pyspark.sql.utils import AnalysisException


In [ ]:
# Required Reporting contract. Repository validation keeps this in exact agreement
# with the six producer declarations, the solution manifest, and active TMDL.
LAKEHOUSE_NAME = "retail_lakehouse"
GOLD_DB = "au"
SILVER_DB = "ag"
MAX_SOURCE_LAG_HOURS = float(os.environ.get("MAX_SOURCE_LAG_HOURS", "24"))
MAX_AS_OF_SKEW_HOURS = float(os.environ.get("MAX_AS_OF_SKEW_HOURS", "6"))
MAX_FORECAST_MAPE = float(os.environ.get("MAX_FORECAST_MAPE", "1.0"))

ML_OUTPUT_CONTRACTS = {'demand_forecast': [('store_id', 'long', False),
                     ('product_id', 'long', False),
                     ('forecast_date', 'date', False),
                     ('predicted_units', 'double', False),
                     ('lower_bound', 'double', False),
                     ('upper_bound', 'double', False),
                     ('mape', 'double', False),
                     ('source_as_of', 'timestamp', False),
                     ('generated_at', 'timestamp', False),
                     ('forecast_horizon_days', 'int', False),
                     ('model_run_id', 'string', False),
                     ('schema_version', 'string', False)],
 'customer_segments': [('customer_id', 'long', False),
                       ('cluster_id', 'int', False),
                       ('segment_label', 'string', False),
                       ('recency_days', 'double', False),
                       ('frequency', 'double', False),
                       ('monetary_value', 'double', False),
                       ('avg_order_value', 'double', False),
                       ('first_purchase_date', 'date', False),
                       ('last_purchase_date', 'date', False),
                       ('segmented_at', 'timestamp', False),
                       ('generated_at', 'timestamp', False),
                       ('model_run_id', 'string', False),
                       ('schema_version', 'string', False)],
 'churn_predictions': [('customer_id', 'long', False),
                       ('churn_probability', 'double', False),
                       ('churn_prediction', 'int', False),
                       ('risk_category', 'string', False),
                       ('is_churned_actual', 'int', True),
                       ('prediction_date', 'timestamp', False),
                       ('generated_at', 'timestamp', False),
                       ('model_version', 'string', False),
                       ('churn_window_days', 'int', False),
                       ('model_run_id', 'string', False),
                       ('schema_version', 'string', False)],
 'stockout_risk': [('store_id', 'long', False),
                   ('product_id', 'long', False),
                   ('risk_id', 'string', False),
                   ('inventory_as_of', 'timestamp', False),
                   ('current_inventory', 'long', False),
                   ('demand_velocity_daily', 'double', False),
                   ('days_of_inventory', 'double', False),
                   ('demand_trend', 'double', False),
                   ('stockout_probability', 'double', False),
                   ('stockout_predicted', 'int', False),
                   ('risk', 'string', False),
                   ('ranking', 'int', False),
                   ('risk_level', 'string', False),
                   ('predicted_at', 'timestamp', False),
                   ('generated_at', 'timestamp', False),
                   ('forecast_horizon_days', 'int', False),
                   ('Department', 'string', True),
                   ('Category', 'string', True),
                   ('Subcategory', 'string', True),
                   ('model_run_id', 'string', False),
                   ('schema_version', 'string', False)],
 'product_recommendations': [('product_id', 'long', False),
                             ('recommended_product_id', 'long', False),
                             ('support', 'double', False),
                             ('confidence', 'double', False),
                             ('lift', 'double', False),
                             ('computed_at', 'timestamp', False),
                             ('generated_at', 'timestamp', False),
                             ('model_run_id', 'string', False),
                             ('schema_version', 'string', False)],
 'price_elasticity': [('product_id', 'long', False),
                      ('product_name', 'string', False),
                      ('product_category', 'string', False),
                      ('elasticity_coefficient', 'double', True),
                      ('elasticity_category', 'string', False),
                      ('confidence_interval_lower', 'double', True),
                      ('confidence_interval_upper', 'double', True),
                      ('standard_error', 'double', True),
                      ('r_squared', 'double', True),
                      ('n_observations', 'long', False),
                      ('sxx', 'double', False),
                      ('sse', 'double', True),
                      ('avg_price', 'double', False),
                      ('regular_price', 'double', False),
                      ('avg_daily_quantity', 'double', False),
                      ('elasticity_evidence_status', 'string', False),
                      ('analysis_design', 'string', False),
                      ('computed_at', 'timestamp', False),
                      ('generated_at', 'timestamp', False),
                      ('model_run_id', 'string', False),
                      ('schema_version', 'string', False)]}

REQUIRED_ML_RULES = {'demand_forecast': {'grain': ('store_id', 'product_id', 'forecast_date'),
                     'as_of': 'generated_at',
                     'lineage': ('source_as_of', 'model_run_id', 'schema_version'),
                     'probabilities': (),
                     'horizon': 'forecast_horizon_days'},
 'customer_segments': {'grain': ('customer_id',),
                       'as_of': 'generated_at',
                       'lineage': ('segmented_at', 'model_run_id', 'schema_version'),
                       'probabilities': (),
                       'horizon': None},
 'churn_predictions': {'grain': ('customer_id',),
                       'as_of': 'generated_at',
                       'lineage': ('prediction_date', 'model_run_id', 'model_version', 'schema_version'),
                       'probabilities': ('churn_probability',),
                       'horizon': None},
 'stockout_risk': {'grain': ('store_id', 'product_id', 'forecast_horizon_days'),
                   'as_of': 'generated_at',
                   'lineage': ('inventory_as_of', 'predicted_at', 'model_run_id', 'schema_version'),
                   'probabilities': ('stockout_probability',),
                   'horizon': None},
 'product_recommendations': {'grain': ('product_id', 'recommended_product_id'),
                             'as_of': 'generated_at',
                             'lineage': ('computed_at', 'model_run_id', 'schema_version'),
                             'probabilities': ('support', 'confidence'),
                             'horizon': None},
 'price_elasticity': {'grain': ('product_id',),
                      'as_of': 'generated_at',
                      'lineage': ('computed_at', 'model_run_id', 'schema_version'),
                      'probabilities': (),
                      'horizon': None}}

# Each producer's source as-of (the latest Silver receipt it read).
SOURCE_AS_OF_COLUMNS = {
    "demand_forecast": "source_as_of",
    "customer_segments": "segmented_at",
    "churn_predictions": "prediction_date",
    "stockout_risk": "predicted_at",
    "product_recommendations": "computed_at",
    "price_elasticity": "computed_at",
}

# Key columns and the Silver dimension (keyed on ID) each must exist in.
FOREIGN_KEYS = {
    "demand_forecast": (("store_id", "dim_stores"), ("product_id", "dim_products")),
    "customer_segments": (("customer_id", "dim_customers"),),
    "churn_predictions": (("customer_id", "dim_customers"),),
    "stockout_risk": (("store_id", "dim_stores"), ("product_id", "dim_products")),
    "product_recommendations": (
        ("product_id", "dim_products"),
        ("recommended_product_id", "dim_products"),
    ),
    "price_elasticity": (("product_id", "dim_products"),),
}

# Category values the producers write and the semantic model's measures filter on.
ALLOWED_CATEGORY_VALUES = {
    "customer_segments": {
        "segment_label": (
            "Champions", "Loyal Customers", "New Customers", "Potential Loyalists",
            "At Risk", "Lost", "Hibernating",
        ),
    },
    "churn_predictions": {
        "risk_category": ("Very Low", "Low", "Medium", "High", "Very High"),
    },
    "stockout_risk": {
        "risk_level": ("HIGH", "MEDIUM", "LOW"),
        "risk": ("HIGH", "MEDIUM", "LOW"),
    },
    "price_elasticity": {
        "elasticity_category": (
            "Highly Elastic", "Elastic", "Unit Elastic", "Inelastic", "Insufficient Evidence",
        ),
    },
}


In [ ]:
def _normalize_ml_type(data_type):
    return data_type.replace("bigint", "long").replace("integer", "int")


def _fail_if(frame, condition, message):
    if frame.filter(condition).limit(1).count():
        raise RuntimeError(message)


def _any_null(columns):
    return reduce(lambda left, right: left | right, (F.col(name).isNull() for name in columns))


def _any_blank(columns):
    return reduce(
        lambda left, right: left | right,
        (F.trim(F.col(name).cast("string")) == "" for name in columns),
    )


def _any_non_finite(columns):
    return reduce(
        lambda left, right: left | right,
        (
            F.isnan(F.col(name))
            | F.col(name).isin(float("inf"), float("-inf"))
            for name in columns
        ),
    )


def _validate_schema(frame, table_name):
    expected = tuple((name, data_type) for name, data_type, _ in ML_OUTPUT_CONTRACTS[table_name])
    actual = tuple(
        (field.name, _normalize_ml_type(field.dataType.simpleString()))
        for field in frame.schema.fields
    )
    if actual != expected:
        raise RuntimeError(
            f"Required ML schema mismatch for {table_name}: expected={expected}, actual={actual}"
        )


def _validate_demand_forecast(frame):
    _fail_if(
        frame,
        (F.col("predicted_units") < 0)
        | (F.col("lower_bound") < 0)
        | (F.col("upper_bound") < F.col("lower_bound"))
        | (F.col("predicted_units") < F.col("lower_bound"))
        | (F.col("predicted_units") > F.col("upper_bound"))
        | (F.col("mape") < 0)
        | (F.col("forecast_horizon_days") <= 0)
        | (F.col("source_as_of") > F.col("generated_at")),
        "demand_forecast contains invalid forecast bounds or horizon values",
    )
    # 06 anchors on the last complete day (the day before the as-of date), so
    # step 1 is the as-of date itself.
    stepped = frame.withColumn(
        "_horizon_step",
        F.datediff("forecast_date", F.to_date("source_as_of")) + 1,
    )
    _fail_if(
        stepped,
        (F.col("_horizon_step") < 1)
        | (F.col("_horizon_step") > F.col("forecast_horizon_days")),
        "demand_forecast contains dates outside its declared horizon",
    )
    horizon = stepped.groupBy("store_id", "product_id").agg(
        F.countDistinct("forecast_date").alias("date_count"),
        F.countDistinct("forecast_horizon_days").alias("horizon_count"),
        F.min("_horizon_step").alias("min_step"),
        F.max("_horizon_step").alias("max_step"),
        F.max("forecast_horizon_days").alias("horizon_days"),
    )
    _fail_if(
        horizon,
        (F.col("horizon_count") != 1)
        | (F.col("date_count") != F.col("horizon_days"))
        | (F.col("min_step") != 1)
        | (F.col("max_step") != F.col("horizon_days")),
        "demand_forecast has an incomplete or inconsistent forecast horizon",
    )


def _validate_customer_segments(frame):
    _fail_if(
        frame,
        (F.col("cluster_id") < 0)
        | (F.col("recency_days") < 0)
        | (F.col("frequency") <= 0)
        | (F.col("monetary_value") <= 0)
        | (F.col("avg_order_value") <= 0)
        | (F.col("first_purchase_date") > F.col("last_purchase_date"))
        | (F.col("segmented_at") > F.col("generated_at")),
        "customer_segments contains invalid RFM values or purchase dates",
    )


def _validate_churn_predictions(frame):
    _fail_if(
        frame,
        (~F.col("churn_prediction").isin(0, 1))
        | F.col("is_churned_actual").isNotNull()
        | (F.col("churn_window_days") <= 0)
        | (F.col("prediction_date") > F.col("generated_at")),
        "churn_predictions contains an invalid class, compatibility value, or horizon",
    )


def _validate_stockout_risk(frame):
    conditions = {
        "invalid_class": ~F.col("stockout_predicted").isin(0, 1),
        "negative_current_inventory": F.col("current_inventory") < 0,
        "negative_demand_velocity": F.col("demand_velocity_daily") < 0,
        "negative_days_of_inventory": F.col("days_of_inventory") < 0,
        "invalid_horizon": F.col("forecast_horizon_days") <= 0,
        "invalid_ranking": F.col("ranking") <= 0,
        "inventory_after_prediction": F.col("inventory_as_of") > F.col("predicted_at"),
        "prediction_after_generation": F.col("predicted_at") > F.col("generated_at"),
    }
    counts = frame.agg(*[
        F.count(F.when(condition, F.lit(1))).alias(name)
        for name, condition in conditions.items()
    ]).first().asDict()
    failures = [f"{name}={count}" for name, count in counts.items() if count]
    if failures:
        raise RuntimeError(
            "stockout_risk contains invalid inventory, class, horizon, or as-of values: "
            + ", ".join(failures)
        )


def _validate_product_recommendations(frame):
    _fail_if(
        frame,
        (F.col("lift") <= 0)
        | (F.col("product_id") == F.col("recommended_product_id"))
        | (F.col("computed_at") > F.col("generated_at")),
        "product_recommendations contains invalid lift, self-pairs, or as-of values",
    )


def _validate_price_elasticity(frame):
    _fail_if(
        frame,
        (F.col("n_observations") < 0)
        | (F.col("avg_price") <= 0)
        | (F.col("regular_price") <= 0)
        | (F.col("avg_daily_quantity") < 0)
        | (F.col("sxx") < 0)
        | (
            F.col("confidence_interval_lower").isNotNull()
            & F.col("confidence_interval_upper").isNotNull()
            & (
                F.col("confidence_interval_lower")
                > F.col("confidence_interval_upper")
            )
        )
        | (
            F.col("r_squared").isNotNull()
            & ((F.col("r_squared") < 0) | (F.col("r_squared") > 1))
        )
        | (F.col("computed_at") > F.col("generated_at")),
        "price_elasticity contains invalid observation counts, prices, "
        "intervals, fit statistics, or as-of values",
    )


def _single_source_as_of(frame, table_name):
    column = SOURCE_AS_OF_COLUMNS[table_name]
    values = [row[0] for row in frame.select(column).distinct().limit(2).collect()]
    if len(values) != 1:
        raise RuntimeError(f"{table_name}.{column} must hold exactly one source as-of value")
    return values[0]


def _validate_source_as_of(as_of_by_table, silver_as_of):
    """Every output describes the same, current Silver snapshot."""
    hours = lambda delta: delta.total_seconds() / 3600.0  # noqa: E731
    stale = {
        table: value for table, value in as_of_by_table.items()
        if hours(silver_as_of - value) > MAX_SOURCE_LAG_HOURS
    }
    if stale:
        raise RuntimeError(
            f"Required ML outputs are more than {MAX_SOURCE_LAG_HOURS}h behind the Silver "
            f"as-of {silver_as_of}: {stale}"
        )
    ahead = {table: value for table, value in as_of_by_table.items() if value > silver_as_of}
    if ahead:
        raise RuntimeError(f"Required ML outputs are ahead of the Silver as-of {silver_as_of}: {ahead}")
    values = list(as_of_by_table.values())
    if values and hours(max(values) - min(values)) > MAX_AS_OF_SKEW_HOURS:
        raise RuntimeError(
            f"Required ML outputs disagree on their source as-of by more than "
            f"{MAX_AS_OF_SKEW_HOURS}h: {as_of_by_table}"
        )


def _validate_foreign_keys(frame, table_name, dimensions):
    for column, dimension in FOREIGN_KEYS[table_name]:
        keys = dimensions[dimension].select(F.col("ID").cast("long").alias(column))
        orphans = frame.select(column).distinct().join(keys, column, "left_anti")
        if orphans.limit(1).count():
            examples = [row[0] for row in orphans.limit(5).collect()]
            raise RuntimeError(
                f"{table_name}.{column} has keys missing from {dimension}: {examples}"
            )


def _validate_categories(frame, table_name):
    for column, allowed in ALLOWED_CATEGORY_VALUES.get(table_name, {}).items():
        _fail_if(
            frame,
            ~F.col(column).isin(*allowed),
            f"{table_name}.{column} holds a value outside {allowed}",
        )


def _validate_quality(frame, table_name):
    if table_name == "demand_forecast":
        _fail_if(
            frame,
            F.col("mape") >= F.lit(MAX_FORECAST_MAPE),
            f"demand_forecast mape is not below {MAX_FORECAST_MAPE} "
            "(no better than forecasting zero)",
        )


TABLE_VALIDATORS = {
    "demand_forecast": _validate_demand_forecast,
    "customer_segments": _validate_customer_segments,
    "churn_predictions": _validate_churn_predictions,
    "stockout_risk": _validate_stockout_risk,
    "product_recommendations": _validate_product_recommendations,
    "price_elasticity": _validate_price_elasticity,
}


def validate_required_table(table_name, dimensions):
    full_name = f"{LAKEHOUSE_NAME}.{GOLD_DB}.{table_name}"
    try:
        frame = spark.table(full_name)
    except AnalysisException as exc:
        raise RuntimeError(f"Required ML table is missing: {full_name}") from exc
    if frame.limit(1).count() == 0:
        raise RuntimeError(f"Required ML table is empty: {full_name}")

    _validate_schema(frame, table_name)
    rule = REQUIRED_ML_RULES[table_name]
    required_values = (*rule["grain"], rule["as_of"], *rule["lineage"])
    _fail_if(
        frame,
        _any_null(required_values),
        f"{table_name} contains null grain/as-of/lineage values",
    )
    non_nullable_outputs = tuple(
        name
        for name, _, nullable in ML_OUTPUT_CONTRACTS[table_name]
        if not nullable and name not in required_values
    )
    if non_nullable_outputs:
        _fail_if(
            frame,
            _any_null(non_nullable_outputs),
            f"{table_name} contains null required output values",
        )
    floating_outputs = tuple(
        name
        for name, data_type, _ in ML_OUTPUT_CONTRACTS[table_name]
        if data_type == "double"
    )
    if floating_outputs:
        _fail_if(
            frame,
            _any_non_finite(floating_outputs),
            f"{table_name} contains NaN or infinite floating-point values",
        )
    _fail_if(
        frame,
        _any_blank(rule["lineage"]),
        f"{table_name} contains blank lineage values",
    )
    duplicates = frame.groupBy(*rule["grain"]).count()
    _fail_if(
        duplicates,
        F.col("count") != 1,
        f"{table_name} contains duplicate grain keys",
    )
    for probability in rule["probabilities"]:
        _fail_if(
            frame,
            F.col(probability).isNull()
            | (F.col(probability) < 0.0)
            | (F.col(probability) > 1.0),
            f"{table_name}.{probability} is null or outside [0, 1]",
        )
    TABLE_VALIDATORS[table_name](frame)
    _validate_categories(frame, table_name)
    _validate_quality(frame, table_name)
    _validate_foreign_keys(frame, table_name, dimensions)
    return frame.count(), _single_source_as_of(frame, table_name)


In [ ]:
def read_silver(table_name):
    full_name = f"{LAKEHOUSE_NAME}.{SILVER_DB}.{table_name}"
    try:
        return spark.table(full_name)
    except AnalysisException as exc:
        raise RuntimeError(f"Silver table is missing: {full_name}") from exc


dimensions = {
    name: read_silver(name) for name in ("dim_stores", "dim_products", "dim_customers")
}
silver_as_of = read_silver("fact_receipts").agg(
    F.max(F.col("event_ts").cast("timestamp"))
).first()[0]
if silver_as_of is None:
    raise RuntimeError("Silver fact_receipts has no rows; there is no source as-of.")

validated_counts = {}
source_as_of_by_table = {}
for required_table in ML_OUTPUT_CONTRACTS:
    validated_counts[required_table], source_as_of_by_table[required_table] = (
        validate_required_table(required_table, dimensions)
    )
_validate_source_as_of(source_as_of_by_table, silver_as_of)

print(f"Required ML Reporting gate passed (Silver as-of {silver_as_of}).")
for table_name, row_count in validated_counts.items():
    print(f"  {table_name}: {row_count:,} rows")
